In [10]:

# quickstart_eval.py
import json
import os
import sys
from pathlib import Path
from urllib import response
import uuid
import litellm

# Resolve the repo root and make it the working directory so that app/config.py
# relative paths (CONVERSATIONS_DB, DATA_PATH, etc.) resolve correctly.
repo_root = Path.cwd()
if not (repo_root / "app").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
os.chdir(repo_root)

# LiteLLM (used by MLflow judge) reads OLLAMA_API_BASE, not OLLAMA_BASE_URL.
if "OLLAMA_API_BASE" not in os.environ:
    _ollama_base = os.getenv("OLLAMA_BASE_URL")
    if _ollama_base:
        os.environ["OLLAMA_API_BASE"] = _ollama_base

from langchain_core.messages import AIMessage
import mlflow
from openai import OpenAI
from mlflow.genai import scorer
from mlflow.genai.scorers import Correctness, Guidelines

from app.config import LLM_MODEL, LLM_BASE_URL
from app.factory import get_judge_model_uri
from app.graph import graph


13:29:45 [graph] Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
13:29:45 [graph] Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
13:29:45 [graph] Using gemini LLM: gemma-3-27b-it


In [2]:
# Evaluation dataset
base_dir = Path.cwd()
repo_root = base_dir if (base_dir / "app").exists() else base_dir.parent
dataset_path = repo_root / "evaluation" / "eval_dataset.json"
# dataset_path = repo_root / "evaluation" / "eval_dataset_converted.json"
with dataset_path.open() as f:
    eval_dataset = json.load(f)

In [8]:
correctness_judge = Correctness(model="ollama_chat:/phi3.5")

# Call it directly like a function
feedback = correctness_judge(
    inputs={"request": "What is MLflow?"},
    outputs={"response": "MLflow is an open-source platform for managing the ML lifecycle."},
    expectations={
        "expected_facts": [
            "MLflow is open-source",
            "MLflow manages the ML lifecycle"
        ]
    }
)

print(feedback.value)     # "yes" or "no"
print(feedback.rationale) # explanation from the judge

11:12:31 - LiteLLM:INFO: utils.py:3919 - 
LiteLLM completion() model= phi3.5; provider = ollama_chat
11:12:31 [graph] 
LiteLLM completion() model= phi3.5; provider = ollama_chat
11:12:39 - LiteLLM:INFO: utils.py:1643 - Wrapper: Completed Call, calling success_handler
11:12:39 [graph] Wrapper: Completed Call, calling success_handler


yes
Let's think step by step: The document provides information directly addressing both parts of the claim. It confirms that MLflow is open-source, which supports the first part of the claim stating 'MLflow is open-source'. Additionally, it states that MLflow manages the machine learning lifecycle, aligning with and supporting the second portion: '- MLflow manages the ML lifecycle.' Since both components of the claim are affirmed by the document within the context of answering what MLflow is, we conclude 'The response is correct'. There's no part that isn’t supported based on this assessment.


In [3]:
# Use different env variable when using a different LLM provider
mlflow.set_experiment("RAG Agent Evaluation 4")

def rag_agent(question: str) -> str:
    config = {"configurable": {"thread_id": str(uuid.uuid4())}}

    result = graph.invoke(
        {"messages": question, "context": [], "retrieved": []},
        config=config,
    )

    answer = ""
    for m in reversed(result["messages"]):
        if isinstance(m, AIMessage):
            answer = m.content
            break

    return answer

# Wrapper function for evaluation
def qa_predict_fn(question: str) -> str:
    out = rag_agent(question)
    return out
# {"response": response}

In [ ]:
# Scorers
@scorer
def is_concise(outputs: str) -> bool:
    return len(outputs.split()) <= 5

scorers = [
    # Correctness(model=get_judge_model_uri()),
    Correctness(model="ollama:/phi3.5"),
    # Correctness(),
    
    # Guidelines(name="is_english", guidelines="The answer must be in English"),
    is_concise,
]

results = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=qa_predict_fn,
    scorers=scorers,
)

2026/03/17 11:01:57 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2026/03/17 11:01:57 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.
11:01:57 [graph] Retrieved 4 chunks in 0.20s
11:01:57 [graph] AFC is enabled with max remote calls: 10.
11:02:10 [graph] Generated answer in 13.43s
Evaluating:   0%|          | 0/1 [Elapsed: 00:00, Remaining: ?] 11:02:11 [graph] Retrieved 4 chunks in 0.14s
11:02:11 [graph] AFC is enabled with max remote calls: 10.
11:02:19 [graph] Generated answer in 7.84s
11:02:19 - LiteLLM:INFO: utils.py:3919 - 
LiteLLM completion() model= phi3.5; provider = ollama
11:02:19 [graph] 
LiteLLM completion() model= phi3.5; provider = ollama
Evaluating:   0%|     

In [17]:
results.tables

{'eval_results':                               trace_id correctness/value  \
 0  tr-dfb3c3cf9e9f46257c20529c9c4ced35           unknown   
 
                              expected_response/value  \
 0  QProv is an automated system that collects and...   
 
                                                trace client_request_id state  \
 0  {"info": {"trace_id": "tr-dfb3c3cf9e9f46257c20...              None    OK   
 
     request_time  execution_duration  \
 0  1773736226993                6247   
 
                                              request  \
 0  {'model': 'gemma-3-27b-it', 'contents': [{'par...   
 
                                             response  \
 0  {'sdk_http_response': {'headers': {'content-ty...   
 
                                       trace_metadata  \
 0  {'mlflow.source.git.repoURL': 'git@github.com:...   
 
                                                 tags  \
 0  {'mlflow.artifactLocation': 'mlflow-artifacts:...   
 
                                

In [21]:
import os
import litellm
import openai
import mlflow
from mlflow.genai.scorers import Correctness, Guidelines
import uuid

# Tell litellm where your Ollama is
os.environ["OLLAMA_API_BASE"] = "http://localhost:11435"

# 1. Define a simple QA dataset
dataset = [
    {
        "inputs": {"question": "Can MLflow manage prompts?"},
        "expectations": {"expected_response": "Yes!"},
    },
    {
        "inputs": {"question": "Can MLflow create a taco for my lunch?"},
        "expectations": {
            "expected_response": "No, unfortunately, MLflow is not a taco maker."
        },
    },
]

In [20]:
dataset = [
    {
        "inputs": {
            "question": "What Qiskit API call should I use to log QProv field Q4 (Circuit Width) to MLflow?"
        },
        "expectations": {
            "expected_response": "You should use circuit.num_qubits to log QProv field Q4 (Circuit Width), and log it with mlflow.log_param('circuit_width', circuit.num_qubits). Do not use circuit.width(), because Qiskit's width() returns the total number of qubits plus classical bits combined, which is broader than the QProv definition. QProv defines circuit width strictly as the number of qubits."
        },
    },
    {
        "inputs": {
            "question": "Why is recording the transpilation random seed important for quantum experiment reproducibility?"
        },
        "expectations": {
            "expected_response": "Recording the transpilation random seed (QProv field C4) is important because transpilation involves randomized algorithms for qubit routing and layout assignment, which is an NP-hard problem. Without a fixed seed, running the same logical circuit through the transpiler on two occasions may produce different physical circuits with different qubit assignments and gate mappings. This means the experiment cannot be exactly reproduced. By logging the seed via mlflow.log_param('transpile_seed', seed) and passing it as seed_transpiler to qiskit.transpile(), the exact compiled circuit can be reconstructed."
        },
    },
]

In [14]:
# response = litellm.completion(
#     model="ollama/phi3.5",
#     messages=[{"role": "user", "content": "Hi"}],
#     api_base="http://localhost:11435",
# )
# response.choices[0].message.content

In [15]:
# response = rag_agent("Hi")
# response

In [16]:
def rag_agent(question: str) -> str:
    config = {"configurable": {"thread_id": str(uuid.uuid4())}}

    result = graph.invoke(
        {"messages": question, "context": [], "retrieved": []},
        config=config,
    )

    answer = ""
    for m in reversed(result["messages"]):
        if isinstance(m, AIMessage):
            answer = m.content
            break

    return answer

# Wrapper function for evaluation
def qa_predict_fn(question: str) -> str:
    response = rag_agent(question)
    return response

In [ ]:
# 2. Define a prediction function to generate responses
def predict_fn(question: str) -> str:
    # response = client.chat.completions.create(
    #     model="gpt-4o-mini", messages=[{"role": "user", "content": question}]
    # )
    response = litellm.completion(
        model="ollama/phi3.5",
        messages=[{"role": "user", "content": question}],
        api_base="http://localhost:11435",
    )
    return response.choices[0].message.content


In [23]:
# 3.Run the evaluation
results = mlflow.genai.evaluate(
    data=dataset,
    predict_fn=qa_predict_fn,
    scorers=[
        # Built-in LLM judge
        # Correctness(model="gemini:/gemma-3-27b-it"),
        Correctness(model="ollama:/phi3.5"),
        # Custom criteria using LLM judge
        Guidelines(name="is_english", guidelines="The answer must be in English"),
    ],
)


2026/03/17 13:44:21 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.
2026/03/17 13:44:21 WARNING mlflow.tracing.fluent: Failed to start span LangGraph: 'NonRecordingSpan' object has no attribute 'context'. For full traceback, set logging level to debug.
13:44:21 [graph] Retrieved 4 chunks in 0.16s
13:44:21 [graph] AFC is enabled with max remote calls: 10.
13:44:25 [graph] Generated answer in 4.04s
Evaluating:   0%|          | 0/2 [Elapsed: 00:00, Remaining: ?] 13:44:26 [graph] Retrieved 4 chunks in 0.20s
13:44:26 [graph] Retrieved 4 chunks in 0.21s
13:44:26 [graph] AFC is enabled with max remote calls: 10.
13:44:26 [graph] AFC is enabled with max remote calls: 10.
13:44:27 [graph] Generated answer in 1.21s
13:44:28 - LiteLLM:INFO: utils.py:3919 - 
LiteLLM completion() model= phi3.5; provider = ollama
13:44:28 [graph] 
LiteLLM comple